# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Full CPU validation reached the data-download cell but a Google Drive per-file download quota permanently blocked /RawData/2017-11-11_05-20-08_NIR/imAmbient.bmp after 3 retries; the run aborted and downstream cells cascaded. The 3-session budget (diag, final, contingency) is exhausted for this invocation. Notebook repaired (imAmbient optional per author code; 4 bounded retries) and ready for resume_run_dir; all evidence archived.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: A photometric stereo-based 3D imaging system using computer vision and deep learning for tracking plant growth

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-650819c551206a400b5a5865a67bb67b)


# PS-Plant photometric-stereo pipeline — reproduction on the author test data

**Paper:** Bernotas, G. et al. (2019). *A photometric stereo-based 3D imaging system using computer vision and deep learning for tracking plant growth.* GigaScience 8(5):giz056. doi:10.1093/gigascience/giz056 (PMC6534809)

**Author software:** PS-Plant-Framework, GPLv3, commit `ff1c44bdab65e0f98584f058b13c24cb8ef348fd`
(https://github.com/g2-bernotas/PS-Plant-Framework).

**Test data:** "Photometric stereo test data set", Edinburgh DataShare handle `10283/3279`
(10 PS-Plant acquisitions of an Arabidopsis plant, ROI files, pre-trained Mask R-CNN weights
`mask_rcnn_leaf.h5`, and the author's expected results).

**Scope (partial demonstration).** This notebook executes the author's full PS-Plant processing
chain on the supplied test acquisitions for **Plant 0** and compares every intermediate and final
product with the author's expected results:

1. adaptive light-source-vector (LSV) generation,
2. raw photometric-stereo solve → surface normals / albedo / shadow image (NPZ),
3. rosette masking,
4. pre-trained Mask R-CNN leaf segmentation (512×512 grayscale frames),
5. particle-filter leaf tracking,
6. rosette and leaf-level trait extraction (CSV).

**Execution status (filled after validation):** executed end-to-end on a fresh **Colab CPU**
runtime. A one-example run verifies the software pipeline, *not* the paper's dataset-level
accuracy claims.

**Legacy environment (important).** The author's software targets Python 3.5 / TensorFlow 1.9 /
Keras 2.2.0. This notebook builds an isolated Python 3.7 environment inside Colab and runs the
**author's pinned code verbatim** in it; the notebook kernel itself only orchestrates and
compares results. The TensorFlow-GPU pin is replaced by CPU TensorFlow 1.15.5 (TF 1.15 GPU
needs CUDA 10.0, unavailable on current Colab); Mask R-CNN inference of ten 512×512 images is
fast enough on CPU. GUI/PyQt4 chrome and camera acquisition (FlyCapture/PyCapture) are out of
scope; each GUI action is mapped to its computational core in the cells below.

**日本語まとめ:** このノートブックは PS-Plant（2019, GigaScience）の処理パイプライン（適応光源ベクトル生成 →
測光ステレオ求解 → ロゼットマスク → Mask R-CNN 葉セグメンテーション → 葉トラッキング → 形質抽出）を，
著者提供テストデータ 10 撮影分で実行し，著者の期待結果と比較します。著者コードは Colab 内の
レガシー Python 3.7 環境（TensorFlow 1.15.5 CPU）でコミット固定のまま実行します。GUI 部分は省略し，
計算本体のみを再現します。実行は CPU ランタイムを想定しています（約 30〜45 分，ダウンロード約 3 GB）。

**References**
- Paper: Bernotas, G. et al. (2019). *A photometric stereo-based 3D imaging system using computer vision and deep learning for tracking plant growth.* GigaScience 8(5):giz056. doi:10.1093/gigascience/giz056 (PMC6534809)
- Code: https://github.com/g2-bernotas/PS-Plant-Framework (commit ff1c44b, GPLv3)
- Test data: https://datashare.is.ed.ac.uk/handle/10283/3279 (Edinburgh DataShare)
- Mask R-CNN: He et al. 2017; Matterport implementation adapted by the authors (repo `MaskRCNN/`).

## Runtime selection and how to run

**Select Runtime → Change runtime type → CPU** (the legacy TensorFlow 1.15 stack does not need a GPU
for this workload, and TF 1.15 GPU binaries require CUDA 10.0 which current Colab drivers cannot
provide). Then **Runtime → Run all**.

Expected cost (measured on 2026-10-06, Colab CPU runtime; see the printed timings in each section):
legacy env creation ≈ 5–10 min, selective download ≈ 2.5–3.5 GB, PS solve + Mask R-CNN ≈ 10–20 min.
Expected total ≈ 30–45 min. Free-tier Colab availability and speed vary.

**日本語:** 「ランタイム → ランタイムのタイプを変更 → CPU」を選択してから「すべてのセルを実行」してください。
レガシー TF 1.15 は CPU で十分動作します。所要 30〜45 分・ダウンロード約 3 GB を見込んでください。

In [ ]:
import os, sys, shutil, platform
print("Main notebook kernel:", platform.python_version(), "|", platform.platform())
tot, free = shutil.disk_usage("/content").total / 2**30, shutil.disk_usage("/content").free / 2**30
import psutil
print("Disk: %.0f GiB total, %.0f GiB free" % (tot, free))
print("RAM: %.1f GiB" % (psutil.virtual_memory().total / 2**30))
assert free > 25, "Need >25 GiB free disk in /content for the legacy env and selective data download."


### 1. Legacy author environment (Python 3.7 / TensorFlow 1.15)

The author's `Requirements.txt` pins the scientific stack (numpy, scikit-image, scipy, pandas,
trackpy, matplotlib, TensorFlow 1.x, Keras 2.2). conda is not preinstalled on current Colab VMs,
so we install the single-binary `micromamba` (tooling only) and create an isolated Python 3.7
env inside the Colab VM, then install verified-compatible pins: `tensorflow==1.15.5` (CPU wheel
of the author's TF 1.15 line; the author pin `tensorflow-gpu==1.15.2` cannot run on Colab),
`keras==2.2.4`, `numpy==1.18.5` (TF-1.15-compatible), `opencv-python==3.4.2.17` (author-era
OpenCV 3.4), `scikit-image==0.14.2` (the numpy-1.16-compatibility bugfix of the author's
0.14.x line; 0.14.1 cannot import with numpy>=1.16 as required by TF 1.15) and
`protobuf==3.20.3` (required by TF 1.15), plus the author's remaining pins. Acquisition-only packages (pyserial,
FlyCapture/PyCapture) and training-only helpers (visvis, imgaug) are not on the executed path
and are omitted. **日本語:** 著者環境（Python 3.7 + TF 1.15 系）を micromamba で Colab 内に構築します。
GPU 用ピンは CPU 版 TF 1.15.5 に置き換え（要 CUDA 10.0 のため），それ以外は著者ピンに準拠します。

In [ ]:
import subprocess, time, os
os.makedirs("/content/tools", exist_ok=True)
# conda is not preinstalled on current Colab VMs; micromamba is a single static binary (tooling only)
subprocess.run(["bash", "-c",
    "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content/tools bin/micromamba"],
    check=True)
t0 = time.time()
env = dict(os.environ, MAMBA_ROOT_PREFIX="/content/mamba")
subprocess.run(["/content/tools/bin/micromamba", "create", "-y", "-p", "/content/psplant-env", "python=3.7"],
               check=True, env=env)
print("legacy env created in %.1f min" % ((time.time() - t0) / 60), flush=True)


Install the pinned packages (streams pip output; ~5–10 min).
**日本語:** 依存パッケージを固定バージョンでインストールします（数分）。

In [ ]:
PKGS = [
    "numpy==1.18.5", "tensorflow==1.15.5", "keras==2.2.4", "h5py==2.10.0", "protobuf==3.20.3",
    "opencv-python==3.4.2.17", "scikit-image==0.14.2", "scipy==1.1.0",
    "pandas==0.23.4", "trackpy==0.4.1", "matplotlib==2.2.0", "retrying==1.3.3",
]
import subprocess, time
t0 = time.time()
subprocess.run(["/content/psplant-env/bin/pip", "install", "--no-input", *PKGS], check=True)
print("pip install done in %.1f min" % ((time.time() - t0) / 60), flush=True)


Verify the legacy environment actually imports the versions we will rely on.
**日本語:** レガシー環境のバージョンを確認します。

In [ ]:
SCRIPT = """
import sys, numpy, cv2, skimage, scipy, pandas, matplotlib
import tensorflow as tf, keras, trackpy
print("python", sys.version.split()[0])
print("TF", tf.__version__, "| keras", keras.__version__, "| cv2", cv2.__version__, "| numpy", numpy.__version__)
print("skimage", skimage.__version__, "| scipy", scipy.__version__, "| pandas", pandas.__version__, "| trackpy", trackpy.__version__)
"""
open("/content/psplant", "a").close() if False else None
import os
os.makedirs("/content/psplant", exist_ok=True)
open("/content/psplant/vercheck.py", "w").write(SCRIPT)
import subprocess
subprocess.run(["/content/psplant-env/bin/python", "/content/psplant/vercheck.py"], check=True)


### 2. Author code, pinned commit (inside Colab only)

We fetch the author's repository at the verified commit `ff1c44b…` with a shallow fetch and a
sparse checkout of the Python package files used by the pipeline (`*.py`, `MaskRCNN/mrcnn/*.py`,
`MaskRCNN/samples/GigaScience/*.py`, `PSConfig.properties`). The checked-out commit is asserted
against the pin. **日本語:** 著者リポジトリをコミット固定・部分チェックアウトで取得します（検証つき）。

In [ ]:
import subprocess
CMDS = r"""
set -euo pipefail
rm -rf /content/PS-Plant-Framework
git clone --filter=blob:none --no-checkout https://github.com/g2-bernotas/PS-Plant-Framework.git /content/PS-Plant-Framework
cd /content/PS-Plant-Framework
git config core.sparseCheckout true
printf "*.py
MaskRCNN/mrcnn/*.py
MaskRCNN/samples/GigaScience/*.py
PSConfig.properties
" > .git/info/sparse-checkout
git fetch --depth=1 origin ff1c44bdab65e0f98584f058b13c24cb8ef348fd
git checkout --detach FETCH_HEAD
echo "checked out: $(git rev-parse HEAD)"
test "$(git rev-parse HEAD)" = "ff1c44bdab65e0f98584f058b13c24cb8ef348fd"
ls MaskRCNN/mrcnn | head -3
"""
subprocess.run(["bash", "-c", CMDS], check=True)


### 3. Test data — provenance and selective download inside Colab

**Canonical archival source:** "Photometric stereo test data set", Edinburgh DataShare handle
`10283/3279` (DSpace 8 item `c2891383-8e1b-4444-997a-3a1d3edfa100`, bitstream
"PS-Plant test data set.zip", 16,652,775,855 bytes — verified 2026-10-06 from the provider's
record metadata). **However, DataShare's edge firewall denies requests from this Colab VM**
(observed HTTP 403 "suspected bot/crawler activity" on every documented route; evidence in the
run report), so we use the **author's own distribution copy**: the pinned repository README links
the test data via `bit.ly/39FwnN7` → public Google Drive folder `TestData`
(`1xm0L-4KL--YjBFM8KMYxJJxWiMTuqCEb`), containing the same `RawData` and `Results` trees.

We enumerate the folder recursively over Drive's public `embeddedfolderview` HTML, select **only
the members this reproduction needs** (raw NIR frames + per-session `PSConfig.properties` +
precomputed NPZs, ambient frames, cropped 512×512 frames, ROI files, the pre-trained weights,
the author-provided masks, and the expected results incl. the adaptive-LSV file), download them
per-file with `gdown`, and verify sizes/counts. `imAmbient.bmp` is optional (the author code
reads it only when a session config sets `subtractAmbient`); Drive per-file download quotas are
transient, so downloads retry with bounded waits and optional files may be skipped. **日本語:** DataShare は Colab から 403 拒否のため，
著者 README がリンクする公開 Google Drive フォルダから必要ファイルのみ `gdown` で取得します
（DataShare 側メタデータは検証済みとして本文に記載）。

In [ ]:
import requests
API = "https://datashare.ed.ac.uk/server/api"
BITSTREAM_UUID = "d8da37d7-dd77-4fd6-b3f0-f28e5c4f954b"
try:
    r = requests.get(API + "/core/bitstreams/" + BITSTREAM_UUID, timeout=60, allow_redirects=True)
    print("DataShare API from Colab:", r.status_code, "|", r.text[:120].replace("\n", " "))
    if r.status_code != 200:
        print("-> DataShare content is not reachable from this VM; using the author's Drive copy.")
except Exception as e:
    print("DataShare unreachable:", type(e).__name__, str(e)[:200])


Enumerate the author's public Drive `TestData` folder recursively (public HTML listing; folder
id taken from the pinned README link). **日本語:** Drive フォルダを再帰的に一覧化します。

In [ ]:
import re, requests, time

TESTDATA_FOLDER = "1xm0L-4KL--YjBFM8KMYxJJxWiMTuqCEb"

def list_folder(fid):
    r = requests.get("https://drive.google.com/embeddedfolderview?id=" + fid + "#list", timeout=60)
    r.raise_for_status()
    out = []
    for m in re.finditer(r'id="entry-([-\w]+)"(.*?)(?=id="entry-|flip-entries>|$)', r.text, re.S):
        iid, block = m.group(1), m.group(2)
        nm = re.search(r'flip-entry-title">([^<]*)</div>', block)
        if not nm: continue
        name = nm.group(1)
        kind = "folder" if "/drive/folders/" in block else ("file" if "/file/d/" in block else "?")
        out.append((name, iid, kind))
    return out

def walk(fid, prefix="", depth=0):
    tree = []
    for name, iid, kind in list_folder(fid):
        path = prefix + "/" + name
        if kind == "folder" and depth < 5:
            tree.append((path, iid, "folder"))
            tree += walk(iid, path, depth + 1)
        else:
            tree.append((path, iid, kind))
    return tree

t0 = time.time()
tree = walk(TESTDATA_FOLDER)
files = [t for t in tree if t[2] == "file"]
print("tree entries: %d (%d files) in %.1fs" % (len(tree), len(files), time.time() - t0))
for p, iid, kind in tree[:24]:
    print("%-7s %s" % (kind, p))
print("   ...")


Select the required members and download them per-file with `gdown` (installed from pip;
Drive may throttle but requires no credentials for public files). Selection patterns mirror the
README layout; counts and sizes are asserted after download. **日本語:** 必要ファイルを選択して `gdown` で
ダウンロードし，件数・サイズを検証します。

In [ ]:
import subprocess, os, time

subprocess.run(["/content/psplant-env/bin/pip", "install", "--no-input", "-q", "gdown"], check=True)

def want(path):
    p = path
    parts = p.split("/")
    if "/RawData/" in p and "_NIR/" in p:
        base = parts[-1]
        if base.startswith("im") and base.endswith(".bmp"): return True
        if base == "PSConfig.properties": return True
        if base == "SNZShadowImAndAlbedo_adaptiveLS.npz": return True
    if "/RawData/Cropped/" in p: return True
    if "mask_rcnn_leaf.h5" in p: return True
    if "/RawData/RosetteMasks/" in p or "/RawData/LeafMasks/" in p or "/RawData/TrackedLeafMasks/" in p: return True
    if p.endswith(("roi.txt", "roiLeaf.txt")) and "/Results/" not in p: return True
    if "/Results/" in p: return True  # expected adaptive LSV, masks, DataExtraction CSVs
    return False

selected = [(p, iid) for p, iid, kind in tree if kind == "file" and want(p)]
print("selected files:", len(selected))

os.makedirs("/content/TestData", exist_ok=True)
t0 = time.time()
for i, (p, iid) in enumerate(selected):
    dest = "/content/TestData" + p
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        continue
    # imAmbient.bmp is optional: the author code only uses it when the session config sets
    # subtractAmbient, and loadIms/psNSource never require it. Drive per-file download quotas
    # are transient, so retry with bounded waits and skip imAmbient if it stays unavailable.
    optional = os.path.basename(p) == "imAmbient.bmp"
    ok = False
    for attempt in range(4):
        r = subprocess.run(["/content/psplant-env/bin/gdown", iid, "-O", dest], capture_output=True, text=True)
        if r.returncode == 0 and os.path.exists(dest) and os.path.getsize(dest) > 0:
            ok = True
            break
        print("gdown attempt %d failed for %s: %s" % (attempt + 1, p, (r.stderr or r.stdout)[-160:]), flush=True)
        time.sleep(30)
    if not ok:
        if optional:
            print("WARNING: optional file unavailable after retries (Drive per-file quota), skipping:", p, flush=True)
            continue
        raise RuntimeError("gdown failed for required file " + p)
    if (i % 20 == 0) or i == len(selected) - 1:
        print("[%d/%d] %s (%.1f MiB) %.1fs" % (i + 1, len(selected), p, os.path.getsize(dest) / 2**20, time.time() - t0), flush=True)
    if (i % 20 == 0) or i == len(selected) - 1:
        print("[%d/%d] %s (%.1f MiB) %.1fs" % (i + 1, len(selected), p, os.path.getsize(dest) / 2**20, time.time() - t0), flush=True)
total = sum(os.path.getsize("/content/TestData" + p) for p, _ in selected)
print("downloaded %d files, %.0f MiB in %.1f min" % (len(selected), total / 2**20, (time.time() - t0) / 60))


Verify the extracted layout: 10 NIR sessions with 4 raw frames + config + precomputed NPZ,
cropped 512×512 frames, ROI files, weights, and the author's expected results.
**日本語:** 展開結果を検証し，10 セッションの構成を確認します。

In [ ]:
import glob, os
RAW = sorted(glob.glob("/content/TestData/**/RawData", recursive=True))[0]
print("RAW =", RAW)
sessions = sorted(os.path.basename(d) for d in glob.glob(os.path.join(RAW, "*_NIR")))
print("sessions (%d):" % len(sessions))
for s in sessions:
    ims = sorted(glob.glob(os.path.join(RAW, s, "im[0-9].bmp")))
    print(" ", s, "| raw frames:", len(ims), "| PSConfig:", os.path.exists(os.path.join(RAW, s, "PSConfig.properties")),
          "| provided NPZ:", os.path.exists(os.path.join(RAW, s, "SNZShadowImAndAlbedo_adaptiveLS.npz")))
for sub in ["Cropped", "RosetteMasks", "LeafMasks", "TrackedLeafMasks"]:
    p = os.path.join(RAW, sub)
    print(sub, "->", len(glob.glob(os.path.join(p, "*"))) if os.path.isdir(p) else "MISSING")
assert len(sessions) == 10, "expected 10 NIR test acquisitions"
h5 = glob.glob("/content/TestData/**/mask_rcnn_leaf.h5", recursive=True)
assert h5, "mask_rcnn_leaf.h5 not downloaded"
print("weights:", h5[0], os.path.getsize(h5[0]) / 2**20, "MiB")


### 4. Input preview — provenance and raw data

The 10 NIR acquisition sessions (`2017-11-11_03-00-08_NIR` …, ~20-min time steps) contain
`im0.bmp`–`im3.bmp` (2048×2048 8-bit), an ambient frame, `PSConfig.properties`, and the
author's precomputed NPZ for comparison.
**日本語:** 10 セッションの入力データ構成を前提に，生画像と較正値を表示します。

Show the four raw NIR frames of the first acquisition (the four 940 nm illuminations captured
in ~125–225 ms per the paper) and the session's calibrated light-source vectors.
**日本語:** 最初のセッションの生 NIR 4 枚と較正された光源ベクトルを表示します。

In [ ]:
import cv2, matplotlib.pyplot as plt, re
s0 = sessions[0]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    im = cv2.imread(os.path.join(RAW, s0, "im%d.bmp" % i), cv2.IMREAD_GRAYSCALE)
    ax.imshow(im, cmap="gray"); ax.set_title("%s im%d %s" % (s0, i, im.shape), fontsize=8); ax.axis("off")
fig.suptitle("Raw NIR photometric-stereo frames (author test acquisition 1) — input, not model output")
plt.show()

props = open(os.path.join(RAW, s0, "PSConfig.properties")).read()
m = re.search(r"LS_NIR\s*=\s*(\[.*?\])", props, re.S)
print("LS_NIR =", m.group(1) if m else "not found")
print(sorted(__import__("re").findall(r"^(lens|applyLightingCompensation|subtractAmbient)\s*=\s*(.+)$", props, re.M)))


Preview the author-provided cropped 512×512 grayscale frame (the Mask R-CNN input), the
provided rosette mask, and the provided leaf masks for that frame — reference annotations
produced by the authors, shown here as *reference*, not as our output.
**日本語:** 著者提供の 512×512 入力，ロゼットマスク，葉マスク（参照）を表示します。

In [ ]:
import glob, cv2, matplotlib.pyplot as plt, os
crop = sorted(glob.glob(os.path.join(RAW, "Cropped", "*grayscale*")))
print("cropped grayscale frames:", [os.path.basename(p) for p in crop])
rmask = sorted(glob.glob(os.path.join(RAW, "RosetteMasks", "*.png")))
lmask = sorted(glob.glob(os.path.join(RAW, "LeafMasks", "*.png")))
tmask = sorted(glob.glob(os.path.join(RAW, "TrackedLeafMasks", "*.png")))
print("provided rosette/leaf/tracked masks:", len(rmask), len(lmask), len(tmask))
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
axes[0].imshow(cv2.imread(crop[0], cv2.IMREAD_GRAYSCALE), cmap="gray"); axes[0].set_title("Cropped 512x512 grayscale (input)"); axes[0].axis("off")
axes[1].imshow(cv2.imread(rmask[0], cv2.IMREAD_GRAYSCALE), cmap="gray"); axes[1].set_title("Provided rosette mask (reference)"); axes[1].axis("off")
axes[2].imshow(cv2.imread(lmask[0])); axes[2].set_title("Provided leaf instance masks (reference)"); axes[2].axis("off")
plt.tight_layout(); plt.show()


### 5. Adaptive light-source vectors (GenerateAdaptiveLSVGUI → `genAdaptiveLSVectorsListComp`)

The GUI's *Process* button computes, for every pixel, per-light source vectors shifted by the
pixel's off-axis position (perspective correction via `LensSensorPixelCalc`) and their pseudo-
inverse — saved as `AdaptiveLSV_r2048x2048_ss8.8x8.8_fl16.npz`. We port `genAdaptiveLSVector`
verbatim (GUI progress calls removed) and add a **behavior-preserving vectorization** that is
first checked against the author's per-pixel function on 500 random pixels (max |Δ| reported),
then used for the full 2048×2048 field with the author's camera parameters (focal length 16 mm,
sensor 8.8×8.8 mm, resolution 2048×2048, geometry from the session `PSConfig.properties`).
**日本語:** 適応光源ベクトルを生成します。著者の逐次関数とベクトル化版を 500 画素で照合し（最大誤差を表示），
一致を確認した上で全画素に適用します。

In [ ]:
SCRIPT = r"""
import sys, time, glob, os
sys.path.insert(0, "/content/PS-Plant-Framework")
import numpy as np
from math import sqrt
from PSConfig import PSConfig
from psutils import getLSFromRunMode, Lens
from LensSensorFOVCalc import LensSensorPixelCalc

# --- author's per-pixel function, verbatim (GenerateAdaptiveLSVGUI.genAdaptiveLSVector) ---
def genAdaptiveLSVector(LS, adjX, adjY):
    currLS=np.zeros(LS.shape)
    for idx,L in enumerate(LS):
        currLS[idx, 0]=L[0]+(adjX)
        currLS[idx, 1]=L[1]+(adjY)
        currLS[idx, 2]=L[2]
    if sqrt(currLS[0,0]**2+currLS[0,1]**2+currLS[0,2]**2) != 1:
        for idx,ls in enumerate(currLS):
            currLS[idx]=ls/sqrt(ls[0]**2+ls[1]**2+ls[2]**2)
    if currLS.shape[0]==3:
        LSi=np.linalg.inv(currLS)
    elif currLS.shape[0]>3:
        LS_t = np.transpose(currLS)
        LSi=np.dot(np.linalg.inv(np.dot(LS_t, currLS)),LS_t)
    return LSi, currLS

def genAdaptiveLSVectors_vectorized(LS, imsz, lens):
    # behavior-preserving vectorization of genAdaptiveLSVectorsListComp; validated below
    z=LS[0][2]*10
    pix_w,pix_h=LensSensorPixelCalc(lens.focalLen, [lens.sensorSize[0],lens.sensorSize[1]], z, [imsz[0], imsz[1]])
    midpointX = np.round(imsz[0] / 2)
    midpointY = np.round(imsz[1] / 2)
    adjX = np.arange(imsz[0]-midpointX, 0-(imsz[0]-midpointX), -1)
    adjY = np.arange(imsz[1]-midpointY, 0-(imsz[1]-midpointY), -1)
    pix_w/=10; pix_h/=10
    [i,j]=np.meshgrid(np.arange(0,imsz[0]), np.arange(0, imsz[1]))
    i=i.flatten(); j=j.flatten()
    P=i.size; N=LS.shape[0]
    currLS=np.zeros((P,N,3))
    currLS[:,:,0]=LS[None,:,0]+(adjX[i]*pix_w)[:,None]
    currLS[:,:,1]=LS[None,:,1]+(adjY[j]*pix_h)[:,None]
    currLS[:,:,2]=LS[None,:,2]
    nrm=np.sqrt((currLS**2).sum(-1))
    norm_mask = nrm[:,0] != 1.0
    currLS[norm_mask] = currLS[norm_mask]/nrm[norm_mask][:,:,None]
    LStLS = np.einsum("pni,pnj->pij", currLS, currLS)
    inv = np.linalg.inv(LStLS)
    LSi = np.einsum("pij,pjn->pin", inv, np.transpose(currLS,(0,2,1)))
    return LSi.reshape([imsz[1],imsz[0], 3, N]), currLS.reshape([imsz[1],imsz[0], N, 3])

RAW = sorted(glob.glob("/content/TestData/**/RawData", recursive=True))[0]
sessions = sorted(os.path.basename(d) for d in glob.glob(os.path.join(RAW, "*_NIR")))
psconfig = PSConfig.fromPropertiesFile(os.path.join(RAW, sessions[0], "PSConfig.properties"))
LS = getLSFromRunMode("NIR", psconfig.LS_VIS, psconfig.LS_NIR, psconfig.LS_VISNIR)
imsz = [2048, 2048]; lens = Lens(16, [8.8, 8.8])
print("LS (author PSConfig LS_NIR):", np.round(np.array(LS), 4).tolist())

# --- equivalence check: vectorized vs author per-pixel function on 500 random pixels ---
rng = np.random.RandomState(0)
z = LS[0][2]*10
pw, ph = LensSensorPixelCalc(lens.focalLen, [lens.sensorSize[0], lens.sensorSize[1]], z, imsz)
pw /= 10.0; ph /= 10.0  # genAdaptiveLSVectorsListComp divides pixel sizes by 10 before use
midX = np.round(imsz[0]/2); midY = np.round(imsz[1]/2)
adjX = np.arange(imsz[0]-midX, 0-(imsz[0]-midX), -1)
adjY = np.arange(imsz[1]-midY, 0-(imsz[1]-midY), -1)
LSi_v, LSV_v = genAdaptiveLSVectors_vectorized(LS, imsz, lens)
maxd_i = maxd_v = 0.0
for p in rng.randint(0, imsz[0]*imsz[1], 500):
    ii, jj = int(p // imsz[1]), int(p % imsz[1])
    LSi_1, LSV_1 = genAdaptiveLSVector(np.array(LS), adjX[ii]*pw, adjY[jj]*ph)
    maxd_i = max(maxd_i, np.abs(LSi_1 - LSi_v[jj, ii]).max())
    maxd_v = max(maxd_v, np.abs(LSV_1 - LSV_v[jj, ii]).max())
print("equivalence check: max |dLSAdaptedi|=%.3e  max |dLSAdapted|=%.3e" % (maxd_i, maxd_v))
assert maxd_i < 1e-8 and maxd_v < 1e-8, "vectorized LSV does not match author per-pixel function"

t0 = time.time()
fname = "AdaptiveLSV_r2048x2048_ss8.8x8.8_fl16.npz"
os.makedirs("/content/TestData/OurResults", exist_ok=True)
np.savez_compressed("/content/TestData/OurResults/" + fname, LSAdaptedi=LSi_v, LSAdapted=LSV_v)
print("saved %s (%.1f MiB) in %.1f min" % (fname, os.path.getsize("/content/TestData/OurResults/" + fname)/2**20, (time.time()-t0)/60))
"""
open("/content/psplant/adaptive_lsv.py", "w").write(SCRIPT)
import subprocess, time
t0 = time.time()
r = subprocess.run(["/content/psplant-env/bin/python", "/content/psplant/adaptive_lsv.py"])
assert r.returncode == 0, "adaptive LSV generation failed"
print("cell elapsed %.1f min" % ((time.time() - t0) / 60))


Compare our adaptive-LSV file with the author's expected `Results/AdaptiveLSV_…npz`.
**日本語:** 生成した適応光源ベクトルを著者の期待ファイルと比較します。

In [ ]:
import glob, numpy as np, os
exp = glob.glob("/content/TestData/**/Results/AdaptiveLSV_*.npz", recursive=True)
assert exp, "expected AdaptiveLSV npz not found in the archive selection"
d_exp = np.load(exp[0]); d_our = np.load("/content/TestData/OurResults/AdaptiveLSV_r2048x2048_ss8.8x8.8_fl16.npz")
print("expected file:", os.path.basename(exp[0]), "| keys:", list(d_exp.keys()), "| shapes:", d_exp["LSAdaptedi"].shape, d_exp["LSAdapted"].shape)
for k in ["LSAdaptedi", "LSAdapted"]:
    a, b = d_exp[k], d_our[k]
    print("%-11s shape %s | max|Δ|=%.3e mean|Δ|=%.3e" % (k, a.shape, np.abs(a - b).max(), np.abs(a - b).mean()))


### 6. Raw PS processing → `SNZShadowImAndAlbedo_adaptiveLS.npz` (BatchProcessRawSessionsGUI)

For each of the 10 NIR sessions the GUI's *Process* action: loads `im0..im3.bmp`, subtracts the
ambient frame if configured, applies the lighting-compensation weight maps, computes the shadow
image, solves per-pixel normals/albedo with the adaptive LSVs (`psNSourceAdaptive`), integrates
the height map with Frankot–Chellappa (`fcint`), and saves the NPZ — replicated here with the
author's functions verbatim. The per-pixel solve is vectorized; it is validated against the
author's `psNSourceAdaptivePL` on 2000 random pixels of the first session before use.
**日本語:** 各セッションの生画像から法線・アルベド・陰影画像・高さを求め NPZ に保存します。逐次版とベクトル化版を
2000 画素で照合してから全セッションに適用します。

In [ ]:
SCRIPT = r"""
import sys, os, glob, time
sys.path.insert(0, "/content/PS-Plant-Framework")
import numpy as np, cv2
from psutils import shadowImage, loadIms, loadProperties, saveData, getLSFromRunMode, PGLens, generateWeightMaps
from PSConfig import PSConfig
from psNSourceAdaptive import psNSourceAdaptivePL
import fcint as fc

RAW = sorted(glob.glob("/content/TestData/**/RawData", recursive=True))[0]
OUR = "/content/TestData/OurResults"
sessions = sorted(os.path.basename(d) for d in glob.glob(os.path.join(RAW, "*_NIR")))
LSAdaptedi = np.load(os.path.join(OUR, "AdaptiveLSV_r2048x2048_ss8.8x8.8_fl16.npz"))["LSAdaptedi"]
N = LSAdaptedi.shape[3]

def solve_vectorized(imgs, LSi, imsz):
    # behavior-preserving vectorization of psNSourceAdaptive (per-pixel psNSourceAdaptivePL)
    P = imsz[0]*imsz[1]
    I = imgs.astype(np.float64)          # (N, P)
    AandN = np.einsum("pij,ip->pj", LSi, I)
    a = np.sqrt((AandN**2).sum(1)); a = a + np.spacing(1)
    normals = np.divide(AandN, a[:, None])
    A = a.reshape(imsz)
    Nx = normals[:, 0].reshape(imsz); Ny = normals[:, 1].reshape(imsz); Nz = normals[:, 2].reshape(imsz)
    Gx = np.divide(Nx, Nz + np.spacing(1)); Gy = np.divide(Ny, Nz + np.spacing(1))
    return Gx, Gy, Nx, Ny, Nz, A

# --- equivalence check on 2000 random pixels of session 0 (author per-pixel function) ---
d0 = os.path.join(RAW, sessions[0])
imgs0, imsz = loadIms(d0, N)
rng = np.random.RandomState(0)
P = imsz[0]*imsz[1]
LSi = LSAdaptedi.reshape(P, LSAdaptedi.shape[2], N)
I0 = np.array(imgs0).astype(np.float64)
_, _, Nxv, Nyv, Nzv, Av = solve_vectorized(I0, LSi, imsz)
maxd = 0.0
for p in rng.randint(0, P, 2000):
    A1, Nx1, Ny1, Nz1 = psNSourceAdaptivePL(I0[:, p], LSi[p])
    maxd = max(maxd, abs(A1-Av.flat[p]), abs(Nx1-Nxv.flat[p]), abs(Ny1-Nyv.flat[p]), abs(Nz1-Nzv.flat[p]))
print("equivalence check psNSourceAdaptivePL vs vectorized: max|d|=%.3e" % maxd)
assert maxd < 1e-8

t00 = time.time()
for s in sessions:
    t0 = time.time()
    d = os.path.join(RAW, s)
    psconfig = loadProperties(d, "PSConfig.properties")
    runMode = s.split("_")[-1]
    LS = getLSFromRunMode(runMode, psconfig.LS_VIS, psconfig.LS_NIR, psconfig.LS_VISNIR)
    imgs, imsz = loadIms(d, N)
    imgs = np.array(imgs)

    ambientIm = []
    if psconfig.subtractAmbient:
        ambientFname = os.path.join(d, "imAmbient.bmp")
        if os.path.exists(ambientFname):
            ambientIm = cv2.imread(ambientFname, cv2.IMREAD_GRAYSCALE)
    if np.size(ambientIm) > 0:
        for im in imgs:
            np.subtract(im, ambientIm.flatten(), im)

    lens = PGLens(psconfig.lens)
    if psconfig.applyLightingCompensation:
        weightMaps = generateWeightMaps(np.multiply(LS, 10), imsz, lens, [])
        imgs = np.multiply(imgs, weightMaps)

    shadowIm, _, _ = shadowImage(imgs)
    shadowIm = np.reshape(shadowIm, imsz)

    Gx, Gy, Nx, Ny, Nz, A = solve_vectorized(imgs, LSi, imsz)
    im = np.zeros((Nx.shape[0], Nx.shape[1], 3), "float32")
    im[:, :, 0] = Nx; im[:, :, 1] = Ny; im[:, :, 2] = Nz
    z = fc.fcint(Nx, Ny, Nz)

    resdir = os.path.join(OUR, s)
    os.makedirs(resdir, exist_ok=True)
    saveData(os.path.join(resdir, "SNZShadowImAndAlbedo_adaptiveLS.npz"), z, A, im, shadowIm)
    print("%s done in %.1fs (A[0,0]=%.3f Nz.mean=%.3f)" % (s, time.time()-t0, A[0,0], Nz.mean()), flush=True)
print("all %d sessions processed in %.1f min" % (len(sessions), (time.time()-t00)/60))
"""
open("/content/psplant/run_ps.py", "w").write(SCRIPT)
import subprocess, time
t0 = time.time()
r = subprocess.run(["/content/psplant-env/bin/python", "/content/psplant/run_ps.py"])
assert r.returncode == 0, "PS solve failed"
print("cell elapsed %.1f min" % ((time.time() - t0) / 60))


Compare our per-session NPZs with the author's precomputed ones (same arrays, elementwise;
`snIm` columns are [Nx, Ny, Nz]). **日本語:** 自分たちで計算した NPZ を著者の事前計算 NPZ と比較します。

In [ ]:
import glob, os, numpy as np
RAW = sorted(glob.glob("/content/TestData/**/RawData", recursive=True))[0]
sessions = sorted(os.path.basename(d) for d in glob.glob(os.path.join(RAW, "*_NIR")))
print("%-30s %10s %10s %10s %10s" % ("session", "A", "Nx", "Nz", "shadowIm"))
for s in sessions:
    e = np.load(os.path.join(RAW, s, "SNZShadowImAndAlbedo_adaptiveLS.npz"))
    o = np.load(os.path.join("/content/TestData/OurResults", s, "SNZShadowImAndAlbedo_adaptiveLS.npz"))
    print("%-30s %10.2e %10.2e %10.2e %10.2e" % (s,
        np.abs(e["A"] - o["A"]).mean(),
        np.abs(e["snIm"][:, :, 0] - o["snIm"][:, :, 0]).mean(),
        np.abs(e["snIm"][:, :, 2] - o["snIm"][:, :, 2]).mean(),
        np.abs(e["shadowIm"] - o["shadowIm"]).mean()))
print("values are mean absolute differences over 2048x2048 pixels.")


### 7. Rosette masks (MaskGenGUI → `MaskSegmenterBasic`, T=70, minSize=300, diskSize=11)

The GUI's mask generation loads the plant ROI via `separateIntoPlots.loadPlant` (our recomputed
NPZ), thresholds the albedo at T=70, removes objects <300 px, combines with an Otsu/2 shadow
threshold, keeps the component nearest the centre (disk-11 cleanup), and writes
`<plant>_<session>_mask.png` — replicated with the author's functions verbatim.
**日本語:** 著者の `MaskSegmenterBasic`（T=70, minSize=300, diskSize=11）でロゼットマスクを生成します。

In [ ]:
SCRIPT = r"""
import sys, os, glob
sys.path.insert(0, "/content/PS-Plant-Framework")
import numpy as np, cv2
import psutils as ps
from separateIntoPlots import loadPlant
from MaskSegmenterBasic import MaskSegmenterBasic

RAW = sorted(glob.glob("/content/TestData/**/RawData", recursive=True))[0]
OUR = "/content/TestData/OurResults"
ROI = None
for cand in glob.glob("/content/TestData/**/roi.txt", recursive=True):
    if "/Results/" not in cand: ROI = cand; break
assert ROI, "roi.txt not found"
rois = np.loadtxt(ROI, dtype=np.uint16)
print("ROI file:", ROI, rois.shape)
sessions = sorted(os.path.basename(d) for d in glob.glob(os.path.join(RAW, "*_NIR")))
os.makedirs(os.path.join(OUR, "RosetteMasks"), exist_ok=True)

for s in sessions:
    resdir = os.path.join(OUR, s)
    os.makedirs(resdir, exist_ok=True)
    for i in range(4):  # loadPlant/shadowImageFromPth read the raw frames from the same dir
        dst = os.path.join(resdir, "im%d.bmp" % i)
        if not os.path.exists(dst):
            os.symlink(os.path.join(RAW, s, "im%d.bmp" % i), dst)
    A, z, snIm, shadowIm = loadPlant(resdir, 0, ROI)
    shadowIm2, minImg, maxImg = ps.shadowImageFromPth(resdir)
    roi = rois[0]
    minImg = minImg[roi[1]:roi[1]+roi[3], roi[0]:roi[0]+roi[2]]
    params = {"im": A, "minImg": minImg, "T": 70, "minSize": 300, "diskSize": 11}
    mask = MaskSegmenterBasic(params).segmentImage()
    mask.dtype = "uint8"
    out = os.path.join(OUR, "RosetteMasks", "0_%s_mask.png" % s)
    cv2.imwrite(out, mask*255)
    print(s, "mask px:", int((mask > 0).sum()), flush=True)
print("rosette masks written.")
"""
open("/content/psplant/run_maskgen.py", "w").write(SCRIPT)
import subprocess, time
t0 = time.time()
r = subprocess.run(["/content/psplant-env/bin/python", "/content/psplant/run_maskgen.py"])
assert r.returncode == 0, "rosette mask generation failed"
print("cell elapsed %.1f min" % ((time.time() - t0) / 60))


Compare our rosette masks with the author-provided ones (IoU per session).
**日本語:** 生成したロゼットマスクを著者提供マスクと IoU で比較します。

In [ ]:
import glob, os, cv2, numpy as np
RAW = sorted(glob.glob("/content/TestData/**/RawData", recursive=True))[0]
sessions = sorted(os.path.basename(d) for d in glob.glob(os.path.join(RAW, "*_NIR")))
print("%-30s %6s" % ("session", "IoU"))
for s in sessions:
    ours = cv2.imread("/content/TestData/OurResults/RosetteMasks/0_%s_mask.png" % s, cv2.IMREAD_GRAYSCALE) > 127
    provs = [p for p in glob.glob(os.path.join(RAW, "RosetteMasks", "*.png")) if s in os.path.basename(p)]
    prov = cv2.imread(provs[0], cv2.IMREAD_GRAYSCALE) > 127
    if prov.shape != ours.shape:
        print("%-30s  shape mismatch %s vs %s" % (s, ours.shape, prov.shape)); continue
    iou = (ours & prov).sum() / max(1, (ours | prov).sum())
    print("%-30s %6.3f" % (s, iou))


### 8. Leaf segmentation — pre-trained Mask R-CNN (LeafSegmentationGUI)

The GUI builds `ArabidopsisLeafSegmentationConfig` from the author's `MaskRCNN/samples/GigaScience`
config, loads the provided pre-trained weights `TestData/RawData/mask_rcnn_leaf.h5` (grayscale
model, 512×512 input), and runs `model.detect` on every cropped grayscale frame, writing colour
instance masks with the GUI's colour map. We replicate this verbatim in the legacy env; the only
adaptations are the device string (`/cpu:0` here; the author hard-codes `/gpu:0`), POSIX-safe
filename handling, and a `uint8` cast before PNG write. **日本語:** 著者提供の学習済み Mask R-CNN
（グレースケール 512×512）で葉インスタンスを検出します。デバイス指定・パス処理・uint8 キャスト以外は
著者コードのままです。

In [ ]:
SCRIPT = r"""
import sys, os, glob, time
os.environ["CUDA_VISIBLE_DEVICES"] = ""
sys.path.insert(0, "/content/PS-Plant-Framework")
sys.path.insert(0, "/content/PS-Plant-Framework/MaskRCNN")
import numpy as np, cv2
import tensorflow as tf
import mrcnn.model as modellib
from samples.GigaScience import LeafSegmentationConfig

config = LeafSegmentationConfig.ArabidopsisLeafSegmentationConfig()
class InferenceConfig(config.__class__):
    GPU_COUNT = 1
    IMAGES_PER_GPU = 1
config = InferenceConfig()
config.display()

DEVICE = "/gpu:0" if tf.test.is_gpu_available() else "/cpu:0"
print("device:", DEVICE)
with tf.device(DEVICE):
    model = modellib.MaskRCNN(mode="inference", model_dir="/content/psplant-model", config=config)
WEIGHTS = sorted(glob.glob("/content/TestData/**/mask_rcnn_leaf.h5", recursive=True))[0]
print("weights:", WEIGHTS, os.path.getsize(WEIGHTS)/2**20, "MiB")
model.load_weights(WEIGHTS, by_name=True)

cmap = np.array([[192, 57, 43], [243, 156, 18], [26, 188, 156], [41, 128, 185], [142, 68, 173], [44, 62, 80],
                [127, 140, 141], [17, 75, 95], [2, 128, 144], [228, 253, 225], [69, 105, 144],
                [244, 91, 105], [91, 192, 235], [253, 231, 76], [155, 197, 61], [229, 89, 52],
                [250, 121, 33], [124, 82, 47], [86, 15, 94], [38, 63, 77], [1, 52, 55], [319, 29, 82],
                [220,131,52], [52, 220, 194], [52, 115, 220], [255, 163, 138], [251, 255, 138], [87, 90, 12],
                [80, 158, 80], [255, 0, 0], [0, 255, 0], [0, 0, 255], [244, 91, 105], [91, 192, 235],
                [253, 231, 76], [155, 197, 61], [229, 89, 52], [250, 121, 33], [124, 82, 47], [86, 15, 94],
                [38, 63, 77], [222,97,127], [195,114,88], [69,160,104], [132,54,243], [163,130,58], [94,234,184],
                [215,187,177], [47,64,102], [36,227,83], [248,57,253], [164,27,80], [78,201,159]], dtype="uint8")

Type = "grayscale"
CROP = sorted(glob.glob("/content/TestData/**/RawData/Cropped", recursive=True))[0]
imgs = sorted(glob.glob(os.path.join(CROP, "*_" + Type + "*")))
print("cropped %s frames: %d" % (Type, len(imgs)))
SAVE = "/content/TestData/OurResults/LeafMasks"
os.makedirs(SAVE, exist_ok=True)
for i, idx in enumerate(imgs):
    t0 = time.time()
    im = cv2.imread(idx)
    results = model.detect([im], verbose=0)
    r = results[0]
    finalMask = np.zeros([np.size(r["masks"], 1), np.size(r["masks"], 0)])
    finalMask2 = np.zeros([np.size(r["masks"], 1), np.size(r["masks"], 0), 3])
    for j in range(0, np.size(r["masks"], 2)):
        alpha = (j + 1) * r["masks"][:, :, j]
        overlap = np.multiply(finalMask, alpha)
        finalMask = finalMask + alpha
        finalMask[overlap > 0] = np.max(alpha)
    for j in range(0, np.size(r["masks"], 2)):
        finalMask2[finalMask == (j + 1), 0] = cmap[j][0]
        finalMask2[finalMask == (j + 1), 1] = cmap[j][1]
        finalMask2[finalMask == (j + 1), 2] = cmap[j][2]
    name = os.path.basename(idx).split(Type)[0]
    cv2.imwrite(os.path.join(SAVE, "{}mask.png".format(name)), finalMask2.astype(np.uint8))
    print("[%d/%d] %s -> %d instances (%.1fs)" % (i+1, len(imgs), name, np.size(r["masks"], 2), time.time()-t0), flush=True)
print("leaf masks written.")
"""
open("/content/psplant/run_leafseg.py", "w").write(SCRIPT)
import subprocess, time
t0 = time.time()
r = subprocess.run(["/content/psplant-env/bin/python", "/content/psplant/run_leafseg.py"])
assert r.returncode == 0, "leaf segmentation failed"
print("cell elapsed %.1f min" % ((time.time() - t0) / 60))


Compare predicted leaf instance masks with the author-provided leaf masks. Instance order from
Mask R-CNN is not fixed, so instances are matched greedily by IoU; per-frame instance counts and
the mean best-IoU are reported. **日本語:** 予測した葉マスクを著者提供マスクと比較します（インスタンス順は
不定のため IoU で対応付け）。

In [ ]:
import glob, os, cv2, numpy as np
RAW = sorted(glob.glob("/content/TestData/**/RawData", recursive=True))[0]
prov = {os.path.basename(p): p for p in glob.glob(os.path.join(RAW, "LeafMasks", "*.png"))}
ours = {os.path.basename(p): p for p in glob.glob("/content/TestData/OurResults/LeafMasks/*.png")}

def inst_masks(path):
    im = cv2.imread(path)
    ids = np.unique(im.reshape(-1, 3), axis=0)
    ids = ids[(ids != 0).any(axis=1)]
    return [((im == c).all(-1)) for c in ids]

print("%-30s %8s %8s %10s" % ("frame", "ours", "provided", "meanIoU"))
for name in sorted(ours):
    if name not in prov: continue
    a, b = inst_masks(ours[name]), inst_masks(prov[name])
    ious = [max(((ma & mb).sum() / max(1, (ma | mb).sum())) for mb in b) if b else 0 for ma in a]
    print("%-30s %8d %8d %10.3f" % (name, len(a), len(b), float(np.mean(ious)) if ious else 0))


### 9. Leaf tracking (TrackingGUI → trackpy particle filter)

The GUI builds a `features` DataFrame of per-frame instance centroids (after morphological
cleaning of each instance mask), links them with `trackpy.predict.NearestVelocityPredict`
(`span`, `memory`, `max displacement`), filters stub trajectories, and recolours each frame's
instances by assigned track ID. Defaults here follow the author GUI (span 0, memory 20,
max displacement 30, min frames 10); the paper additionally reports tuned parameters (span 10,
radius 30, memory 3, filter 100) — noted, not substituted. Input: **our** predicted leaf masks.
**日本語:** 自分たちの葉マスクから重心特徴を抽出し，trackpy の粒子フィルタでトラッキングします。パラメータは
GUI 既定値（span 0, memory 20, maxDisp 30, minFrames 10）です。

In [ ]:
SCRIPT = r"""
import sys, os, glob
sys.path.insert(0, "/content/PS-Plant-Framework")
import numpy as np, cv2, pandas as pd
import trackpy as tp, trackpy.predict
from skimage.morphology import binary_opening
from skimage.measure import label, regionprops

cmap = np.array([[192, 57, 43], [243, 156, 18], [26, 188, 156], [41, 128, 185], [142, 68, 173], [44, 62, 80],
                [127, 140, 141], [17, 75, 95], [2, 128, 144], [228, 253, 225], [69, 105, 144],
                [244, 91, 105], [91, 192, 235], [253, 231, 76], [155, 197, 61], [229, 89, 52],
                [250, 121, 33], [124, 82, 47], [86, 15, 94], [38, 63, 77], [1, 52, 55], [319, 29, 82],
                [220,131,52], [52, 220, 194], [52, 115, 220], [255, 163, 138], [251, 255, 138], [87, 90, 12],
                [80, 158, 80], [255, 0, 0], [0, 255, 0], [0, 0, 255], [222,97,127], [195,114,88], [69,160,104], [132,54,243], [163,130,58],
                [94,234,184],[215,187,177], [47,64,102], [36,227,83], [248,57,253], [164,27,80], [78,201,159], [187, 17, 77],
                [113, 134, 97], [64, 143, 31], [0, 30, 91], [172, 97, 108], [149, 236, 84], [161, 64, 190], [120, 160, 235],
                [94, 2, 230], [112, 96, 25], [59, 0, 243], [133, 57, 8], [220, 206, 119], [55, 20, 170], [100, 39, 137],
                [19, 99, 10], [161, 126, 162], [220, 0, 129], [67, 87, 124], [232, 61, 51]], dtype="uint8")

LEAF = "/content/TestData/OurResults/LeafMasks"
SAVE = "/content/TestData/OurResults/TrackedLeafMasks"
os.makedirs(SAVE, exist_ok=True)
dirs = sorted(glob.glob(os.path.join(LEAF, "*_mask*")))
print("mask frames:", len(dirs))

# author's UsePKL=False path, verbatim (features DataFrame per frame)
features = pd.DataFrame()
for i, idx in enumerate(dirs):
    im = cv2.imread(idx, 0)
    masks = []
    for j in range(1, 255):
        tmp = np.zeros([np.size(im, 1), np.size(im, 0)])
        tmp[im == j] = 1
        tmp = binary_opening(tmp)
        if(np.sum(tmp) > 50):
            masks.append(tmp)
    mask = np.zeros([np.size(im, 1), np.size(im, 0)])
    for j in range(0, np.size(masks, 0)):
        alpha = (j+1) * masks[j]
        overlap = np.multiply(mask, alpha)
        mask = mask + alpha
        mask[overlap > 0] = np.max(alpha)
    l = label(mask)
    rp = regionprops(l)
    for region in rp:
        features = features.append([{"area": region.area,
                                     "y": region.centroid[0],
                                     "x": region.centroid[1],
                                     "frame": i}])
print("features:", features.shape)

span, memory, maxDisp, minFrames = 0, 20, 30, 10  # author GUI defaults
pred = tp.predict.NearestVelocityPredict(span=span)
t = pred.link_df(features, maxDisp, memory=memory)
t1 = tp.filter_stubs(t, minFrames)
print("tracks:", t1["particle"].nunique())
pd.to_pickle(t1, os.path.join(SAVE, "trajectories.pkl"))

# recolour by track ID (author algorithm; positional iloc replaced by named columns)
for i, idx in enumerate(dirs):
    leaves = cv2.imread(idx, 0)
    im = np.zeros([np.size(leaves, 1), np.size(leaves, 0), 3])
    rows = t1[t1["frame"] == i]
    for _, row in rows.iterrows():
        x, y = int(row["x"]), int(row["y"])
        colour = leaves[y, x]
        minr = 1
        while colour == 0:
            for p in range(-minr, minr+1, 1):
                for c in range(-minr, minr+1, 1):
                    yy, xx = y + p, x + c
                    if 0 <= yy < leaves.shape[0] and 0 <= xx < leaves.shape[1]:
                        colour = leaves[yy, xx]
                        if colour != 0: break
                if colour != 0: break
            minr = minr + 1
        im[leaves == colour] = cmap[int(row["particle"]) % len(cmap)]
    cv2.imwrite(os.path.join(SAVE, os.path.basename(idx)), im.astype(np.uint8))
    if i % 3 == 0: print("frame", i, "recoloured", flush=True)
print("tracked masks written.")
"""
open("/content/psplant/run_tracking.py", "w").write(SCRIPT)
import subprocess, time
t0 = time.time()
r = subprocess.run(["/content/psplant-env/bin/python", "/content/psplant/run_tracking.py"])
assert r.returncode == 0, "tracking failed"
print("cell elapsed %.1f min" % ((time.time() - t0) / 60))


Compare our tracked leaf masks with the author's provided `TrackedLeafMasks` (per-frame
best-IoU instance matching; track IDs themselves are not comparable across runs).
**日本語:** トラッキング結果を著者提供の TrackedLeafMasks と比較します（トラック ID は実行毎に異なるため
インスタンス IoU で評価）。

In [ ]:
import glob, os, cv2, numpy as np
RAW = sorted(glob.glob("/content/TestData/**/RawData", recursive=True))[0]
prov = {os.path.basename(p): p for p in glob.glob(os.path.join(RAW, "TrackedLeafMasks", "*.png"))}
ours = {os.path.basename(p): p for p in glob.glob("/content/TestData/OurResults/TrackedLeafMasks/*.png")}

def inst_masks(path):
    im = cv2.imread(path)
    ids = np.unique(im.reshape(-1, 3), axis=0)
    ids = ids[(ids != 0).any(axis=1)]
    return [((im == c).all(-1)) for c in ids]

print("%-30s %8s %8s %10s" % ("frame", "ours", "provided", "meanIoU"))
for name in sorted(ours):
    if name not in prov: continue
    a, b = inst_masks(ours[name]), inst_masks(prov[name])
    ious = [max(((ma & mb).sum() / max(1, (ma | mb).sum())) for mb in b) if b else 0 for ma in a]
    print("%-30s %8d %8d %10.3f" % (name, len(a), len(b), float(np.mean(ious)) if ious else 0))


### 10. Rosette and leaf trait extraction (GenerateResultsGUI)

The GUI's *Process* action loads each session through `loadPlant` (our recomputed NPZ, ROI
plant 0), computes rosette 2D/3D area, perimeter, circularity, compactness, diameter and mean
elevation (`estimateRosetteArea`, pixel size from the camera geometry), and for the tracked
leaf masks computes blade/petiole statistics (`getLeafNormals`, `getStats`), writing the same
CSV formats. Two documented POSIX fixes: the author's lowercase `'psconfig.properties'` path is
corrected to `'PSConfig.properties'`, and float64 mask arrays are cast to uint8 before
`cv2.imwrite`. **日本語:** `loadPlant` で各セッションを読み込み，ロゼット面積等と葉の blade/petiole 形質を
CSV に出力します（Linux 向けの最小限の修正 2 点を明記）。

In [ ]:
SCRIPT = r"""
import sys, os, glob, math, csv
sys.path.insert(0, "/content/PS-Plant-Framework")
import numpy as np, cv2
import psutils as ps
from separateIntoPlots import loadPlant
from skimage.morphology import disk, binary_opening, remove_small_objects
from skimage.measure import label, regionprops, find_contours
from scipy.spatial.distance import cdist, pdist, squareform
from PIL import Image
from LensSensorFOVCalc import LensSensorPixelCalc
from LeafRegionProps import LeafRegionProps
from PSConfig import PSConfig

RAW = sorted(glob.glob("/content/TestData/**/RawData", recursive=True))[0]
OUR = "/content/TestData/OurResults"
ROI = None
for cand in glob.glob("/content/TestData/**/roi.txt", recursive=True):
    if "/Results/" not in cand: ROI = cand; break

# --- verbatim ports from GenerateResultsGUI (GUI progress/dialog calls removed) ---
def estimateRosetteArea(mask, Nz, camHeight=250, focalLength=25, cam=None):
    if cam is None:
        cam = ps.PGLens(focalLength)
    h, v = LensSensorPixelCalc(cam.focalLen, cam.sensorSize, camHeight, [2048, 2048])
    areaOfOnePixel = h*v
    howManyPixels = np.count_nonzero(mask)
    area2D = howManyPixels*areaOfOnePixel
    areaNz = (1/Nz)*areaOfOnePixel
    areaNz = np.multiply(areaNz, mask > 0)
    areaNz = np.nansum(areaNz)
    return area2D, areaNz, h, v

def calcDistBetween2Points(x0, y0, x1, y1):
    return math.hypot(x1-x0, y1-y0)

def getLengthAndElevation(startCoord, endCoord, sn, mask, px_h, type=LeafRegionProps.PETIOLE):
    if type == LeafRegionProps.PETIOLE:
        Nx = sn[:, :, 0]; Ny = sn[:, :, 1]; Nz = sn[:, :, 2]
        snmask = (Nx == 0) & (Ny == 0) & (Nz == 0)
        Nx[snmask] = np.nan; Ny[snmask] = np.nan; Nz[snmask] = np.nan
        Nx[~mask] = np.nan; Ny[~mask] = np.nan; Nz[~mask] = np.nan
        az, elev, r = ps.cart2sph2(Nx, Ny, Nz)
        meanElev = np.nanmean(elev)
        d = calcDistBetween2Points(startCoord[0], startCoord[1], endCoord[0], endCoord[1])
        h = (d*px_h) / math.cos(math.radians(90)-meanElev)
        return 90-math.degrees(meanElev), h
    else:
        x = np.array([startCoord[0], endCoord[0]])
        y = np.array([startCoord[1], endCoord[1]])
        coefficients = np.polyfit(x, y, 1)
        polynomial = np.poly1d(coefficients)
        x_axis = np.linspace(x.min(), x.max(), (x.max())-(x.min()))
        y_axis = polynomial(x_axis)
        Nx = sn[y_axis.astype(np.int), x_axis.astype(np.int), 0]
        Ny = sn[y_axis.astype(np.int), x_axis.astype(np.int), 1]
        Nz = sn[y_axis.astype(np.int), x_axis.astype(np.int), 2]
        az, elev, r = ps.cart2sph2(Nx, Ny, Nz)
        meanElev = np.nanmean(elev)
        d = calcDistBetween2Points(startCoord[0], startCoord[1], endCoord[0], endCoord[1])
        h = (d*px_h) / math.cos(math.radians(90)-meanElev)
        return 90-math.degrees(meanElev), h

def estimateCentroidZ(petDist, petElev, bladeInsertionCoord, bladeElev, centroid, px_h):
    heightBIP = petDist * math.sin(math.radians(petElev))
    d = calcDistBetween2Points(centroid[0], centroid[1], bladeInsertionCoord[0], bladeInsertionCoord[1]) * px_h
    heightCentroidFromBIP = d * math.tan(math.radians(bladeElev))
    return heightBIP + heightCentroidFromBIP

def getStats(leaves, leafMasks, bladeMasks, stemMasks, labels, h, v):
    plantRegionProps = {}
    areaOfOnePixel = h*v
    for i, l in enumerate(leaves):
        if not np.any(bladeMasks[i]):
            continue
        Nx = l[:, :, 0]; Ny = l[:, :, 1]; Nz = l[:, :, 2]
        leafRP = LeafRegionProps()
        leafRP.label = labels[i]
        leafRP.h = h; leafRP.v = v
        leafRP.area3D = np.nansum(bladeMasks[i] * (areaOfOnePixel * (1/Nz)))
        leafRP.area2D = np.sum(bladeMasks[i] * areaOfOnePixel)
        az, elev, r = ps.cart2sph2(Nx, Ny, Nz)
        elev[bladeMasks[i] == False] = np.nan
        leafRP.meanBladeInclination = 90-np.degrees(np.nanmean(elev))
        leafRP.medianBladeInclination = 90-np.degrees(np.nanmedian(elev))
        bladeRP = regionprops(bladeMasks[i].astype(np.int32))
        if len(bladeRP) != 1:
            print("Too many blades!"); assert len(bladeRP) == 1
        leafRP.bladeWidth2D = bladeRP[0].minor_axis_length * h
        leafRP.bladeLength2D = bladeRP[0].major_axis_length * v
        leafRP.centroid = bladeRP[0].centroid
        petioleMask = stemMasks[i].copy()
        petioleMask = remove_small_objects(petioleMask, 32)
        petioleRP = regionprops(petioleMask.astype(np.int32))
        if petioleRP != []:
            leafRP.petioleWidth2D = petioleRP[0].minor_axis_length * h
            leafRP.petioleLength2DRP = petioleRP[0].major_axis_length * v
        else:
            leafRP.petioleWidth2D = 0
            leafRP.petioleLength2DRP = 0
        if np.any(petioleMask):
            petioleCoords = regionprops(label(petioleMask))[0].coords
            bladeCoords = bladeRP[0].coords
            Y = cdist(bladeCoords, petioleCoords, "euclidean")
            matchLocations = np.where(Y == Y.min())
            coords = petioleCoords[matchLocations[1]]
            meanPetioleCoord = [np.mean(coords[:, 0]), np.mean(coords[:, 1])]
            coords = bladeCoords[matchLocations[0]]
            meanBladeCoord = [np.mean(coords[:, 0]), np.mean(coords[:, 1])]
            bladeInsertionCoord = [np.mean([meanPetioleCoord[0], meanBladeCoord[0]]), np.mean([meanPetioleCoord[1], meanBladeCoord[1]])]
            leafRP.bladeInsertionCoord = [bladeInsertionCoord[1], bladeInsertionCoord[0]]
            az, elev, r = ps.cart2sph2(Nx, Ny, Nz)
            elev[petioleMask == False] = np.nan
            leafRP.meanPetioleInclination = 90-np.degrees(np.nanmean(elev))
            leafRP.medianPetioleInclination = 90-np.degrees(np.nanmedian(elev))
            d = cdist(np.array([bladeInsertionCoord]), bladeCoords)
            matchLocations = np.where(d == d.max())
            bladeTipCoord = np.mean(bladeCoords[matchLocations[1]], axis=0)
            leafRP.bladeTipCoord = [bladeTipCoord[1], bladeTipCoord[0]]
            leafRP.bladeLengthPointBased2D = calcDistBetween2Points(leafRP.bladeInsertionCoord[0], leafRP.bladeInsertionCoord[1], leafRP.bladeTipCoord[0], leafRP.bladeTipCoord[1]) * h
            bladeElev, bladeDist = getLengthAndElevation(np.array(leafRP.bladeInsertionCoord), np.array(leafRP.bladeTipCoord), l, bladeMasks[i], h, type=LeafRegionProps.BLADE)
            leafRP.bladeLengthPointBased3D = bladeDist
            leafRP.pointBasedBladeInclination = bladeElev
            d = cdist(np.array([bladeInsertionCoord]), petioleCoords)
            matchLocations = np.where(d == d.max())
            petioleStartCoord = np.mean(petioleCoords[matchLocations[1]], axis=0)
            leafRP.petioleStartCoord = [petioleStartCoord[1], petioleStartCoord[0]]
            leafRP.petioleLength2D = calcDistBetween2Points(leafRP.petioleStartCoord[0], leafRP.petioleStartCoord[1], leafRP.bladeInsertionCoord[0], leafRP.bladeInsertionCoord[1]) * h
            petElev, petDist = getLengthAndElevation(np.array(leafRP.petioleStartCoord), np.array(leafRP.bladeInsertionCoord), l, petioleMask, h, type=LeafRegionProps.PETIOLE)
            leafRP.petioleLength3D = petDist
            leafRP.pointBasedPetioleInclination = petElev
            leafRP.centroidZ = estimateCentroidZ(petDist, petElev, leafRP.bladeInsertionCoord, bladeElev, leafRP.centroid, h)
            leafRP.bladeTipZ = estimateCentroidZ(petDist, petElev, leafRP.bladeInsertionCoord, bladeElev, leafRP.bladeTipCoord, h)
        plantRegionProps.update({leafRP.label: leafRP})
    return plantRegionProps

def getLeafNormals(sn, labels):
    leaves = []; bladeMasks = []; stemMasks = []; leafMasks = []; leafLabels = []
    maxLabel = labels.max()
    for i in range(1, maxLabel+1):
        leafMask = labels == i
        maskedLeaf = np.zeros(sn.shape)
        maskedLeaf[labels == i, :] = sn[labels == i, :]
        rp = regionprops(leafMask.astype(int))
        if rp != []:
            if rp[0].area < 300: disksize = 5
            elif rp[0].area < 1000: disksize = 7
            elif rp[0].area < 1500: disksize = 9
            elif rp[0].area < 2400: disksize = 11
            else: disksize = 15
            selem = disk(disksize)
            bladeMask = binary_opening(leafMask, selem)
            bladeMasks.append(bladeMask.copy())
            stemMask = leafMask ^ bladeMask
            stemMasks.append(stemMask)
            leafLabels.append(i)
            leaves.append(maskedLeaf.copy())
            leafMasks.append(leafMask.copy())
    return leaves, leafMasks, bladeMasks, stemMasks, leafLabels

def convert2IndexedLabels(labelMask, labelDict):
    palette = labelMask.getpalette()
    indexed = np.array(labelMask)
    num_colours = int(len(palette)/3)
    max_val = float(np.iinfo(indexed.dtype).max)
    map = np.array(palette).reshape(num_colours, 3) / max_val
    uLabels = np.unique(indexed)
    indexedNew = np.zeros(indexed.shape) - 1
    for u in uLabels:
        rgb = map[u]
        if not tuple(rgb) in labelDict:
            if u in list(labelDict.values()):
                maxLabel = np.max(list(labelDict.values())) + 1
                u = maxLabel
            labelDict.update({tuple(rgb): u})
        indexedNew[indexed == u] = labelDict[tuple(rgb)]
    blackLabel = labelDict[tuple(np.array([0., 0., 0.]))]
    if blackLabel > 0:
        indexedNew += 1
        indexedNew[indexedNew == blackLabel+1] = 0
    return indexedNew.astype(np.uint8), labelDict

def dumpRA2CSV(ra, name):
    with open("{}.csv".format(name), "w") as csv_file:
        writer = csv.writer(csv_file, delimiter=",")
        header = ["Session", "2D Area (mm^2)", "3D Area (mm^2)", "Perimeter (mm)", "Circularity", "Compactness", "Diameter (mm)", "Mean Elevation"]
        writer.writerow(header)
        for r in ra:
            row = [r[7], r[0], r[1], r[2], r[3], r[4], r[5], r[6]]
            writer.writerow(row)

def dumpLeaves2CSV(leaves, name):
    with open("{}.csv".format(name), "w") as csv_file:
        writer = csv.writer(csv_file, delimiter=",")
        header = ["Session", "Leaf label", "Blade 2D Area (mm^2)", "Blade 3D Area (mm^2)",
        "Mean Blade Elevation", "Median Blade Elevation", "Point Based Blade Elevation",
        "Blade Length 2D (mm)", "Blade Width 2D (mm)", "Blade Length Point Based 2D (mm)",
        "Blade Length Point Based 3D (mm)", "Petiole Length 2D", "Petiole Length 2D RP",
        "Petiole Width 2D", "Mean Petiole Elevation", "Median Petiole Elevation", "Point Based Petiole Elevation",
        "Petiole Start Coord", "Blade Insertion Point", "Blade Tip Coord", "Centroid", "CentroidZ"]
        writer.writerow(header)
        for s in leaves:
            for k, val in s[1].items():
                row = [s[0], s[1][k].label, s[1][k].area2D, s[1][k].area3D, s[1][k].meanBladeInclination, s[1][k].medianBladeInclination, s[1][k].pointBasedBladeInclination,
                s[1][k].bladeLength2D, s[1][k].bladeWidth2D, s[1][k].bladeLengthPointBased2D, s[1][k].bladeLengthPointBased3D,
                s[1][k].petioleLength2D, s[1][k].petioleLength2DRP, s[1][k].petioleWidth2D, s[1][k].meanPetioleInclination,
                s[1][k].medianPetioleInclination, s[1][k].pointBasedPetioleInclination, s[1][k].petioleStartCoord,
                s[1][k].bladeInsertionCoord, s[1][k].bladeTipCoord, s[1][k].centroid, s[1][k].centroidZ]
                writer.writerow(row)

# --- driver: GenerateResultsGUI.process() port, plant 0 ---
sessions = sorted(os.path.basename(d) for d in glob.glob(os.path.join(RAW, "*_NIR")))
psconfig = PSConfig.fromPropertiesFile(os.path.join(RAW, sessions[0], "PSConfig.properties"))  # POSIX fix: author used lowercase name
lens = ps.PGLens(int(psconfig.lens))
camHeight = psconfig.LS_VIS[0][2] * 10
imsz = cv2.imread(os.path.join(RAW, sessions[0], "im0.bmp"), cv2.IMREAD_GRAYSCALE).shape
h, v = LensSensorPixelCalc(lens.focalLen, lens.sensorSize, camHeight, imsz)
print("pixel size: h=%.6f v=%.6f mm | camHeight=%.1f" % (h, v, camHeight))

outDir = os.path.join(OUR, "DataExtraction")
os.makedirs(outDir, exist_ok=True)
raAll, leafStatsAll, labelsDicts = [], [], {0: {}}
for n, s in enumerate(sessions):
    p = 0
    resdir = os.path.join(OUR, s)
    A, z, snIm, shadowIm = loadPlant(resdir, p, ROI)
    maskFname = os.path.join(OUR, "TrackedLeafMasks", "{}_{}_mask.png".format(p, s))
    mask = cv2.imread(maskFname)
    if mask is None:
        raise Exception("Mask file {} not found.".format(maskFname))
    if len(np.unique(mask)) == 2:
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY); mask = mask.astype(bool)
    else:
        labelMask = cv2.imread(maskFname)
        im = Image.fromarray(labelMask)
        labelMask = im.convert(mode="P", palette=Image.ADAPTIVE)
        labelMask, labelsDicts[p] = convert2IndexedLabels(labelMask, labelsDicts[p])
        leaves, leafMasks, bladeMasks, stemMasks, processedLeafLabels = getLeafNormals(snIm, labelMask)
        plantLeavesStats = getStats(leaves, leafMasks, bladeMasks, stemMasks, processedLeafLabels, h, v)
        leafStatsAll.append([s, plantLeavesStats])
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY); mask = mask.astype(bool)
    Nx = snIm[:, :, 0]; Ny = snIm[:, :, 1]; Nz = snIm[:, :, 2]
    area2D, areaNz, h, v = estimateRosetteArea(mask, Nz, camHeight, lens.focalLen)
    print("%s 2D area: %.2f, 3D area: %.2f" % (s, area2D, areaNz), flush=True)
    tmp = mask.astype(int)
    props = regionprops(tmp)[0]
    p_rw = props.perimeter * h
    circ = (4 * math.pi * props.area) / (props.perimeter * props.perimeter)
    comp = props.solidity
    coords = find_contours(mask, .5)[0]
    D = pdist(coords); D = squareform(D)
    N, _ = np.nanmax(D)*h, np.unravel_index(np.argmax(D), D.shape)
    az, elev, r = ps.cart2sph2(Nx, Ny, Nz)
    elev = (90-np.degrees(elev.copy()))
    elev[mask == False] = np.nan
    meanElev = np.nanmean(elev)
    raAll.append([area2D, areaNz, p_rw, circ, comp, N, meanElev, s])

dumpRA2CSV(raAll, os.path.join(outDir, "repro_0_rosette"))
if len(leafStatsAll):
    dumpLeaves2CSV(leafStatsAll, os.path.join(outDir, "repro_0_leaves"))
print("CSV files written to", outDir)
"""
open("/content/psplant/run_results.py", "w").write(SCRIPT)
import subprocess, time
t0 = time.time()
r = subprocess.run(["/content/psplant-env/bin/python", "/content/psplant/run_results.py"])
assert r.returncode == 0, "trait extraction failed"
print("cell elapsed %.1f min" % ((time.time() - t0) / 60))


Compare our extracted rosette traits with the author's expected `DataExtraction` CSV
(2D/3D area, mm²) and plot both series across the 10 acquisitions. This graph is generated by
this notebook from the actual processed data — it is not a figure from the paper.
**日本語:** 抽出したロゼット形質（2D/3D 面積 mm²）を著者の期待 CSV と比較し，時系列グラフとして表示します
（このグラフは本ノートブックが生成した追加可視化であり，論文の図ではありません）。

In [ ]:
import glob, os, pandas as pd, numpy as np, matplotlib.pyplot as plt
exp = glob.glob("/content/TestData/**/Results/DataExtraction/*rosette*.csv", recursive=True)
assert exp, "expected rosette CSV not found"
e = pd.read_csv(exp[0])
o = pd.read_csv("/content/TestData/OurResults/DataExtraction/repro_0_rosette.csv")
cols = ["2D Area (mm^2)", "3D Area (mm^2)"]
print("expected:"); display(e[["Session"] + cols])
print("ours:"); display(o[["Session"] + cols])
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(o))
ax.plot(x, o["3D Area (mm^2)"], "o-", label="ours: 3D area")
ax.plot(x, e["3D Area (mm^2)"], "s--", label="expected: 3D area")
ax.plot(x, o["2D Area (mm^2)"], "^-", label="ours: 2D area (projected)")
ax.set_xlabel("test acquisition (time series)"); ax.set_ylabel("rosette projected area (mm$^2$)")
ax.set_title("Rosette area across the 10 test acquisitions (notebook-generated comparison)")
ax.legend(); plt.tight_layout(); plt.show()
print("mean |3D area ours - expected| = %.2f mm^2" % np.abs(o["3D Area (mm^2)"].values - e["3D Area (mm^2)"].values).mean())


Display the leaf-level CSV from the tracked masks (blade/petiole traits, one row per leaf and
session). **日本語:** トラッキング済みマスクから抽出した葉レベル形質 CSV を表示します。

In [ ]:
import glob, pandas as pd
leaf_csv = glob.glob("/content/TestData/OurResults/DataExtraction/*_leaves.csv")
if leaf_csv:
    d = pd.read_csv(leaf_csv[0])
    show = ["Session", "Leaf label", "Blade 2D Area (mm^2)", "Blade 3D Area (mm^2)", "Mean Blade Elevation", "Blade Length 2D (mm)"]
    display(d[show].head(12))
    print("rows:", len(d))
else:
    print("No leaf CSV was produced (see trait-extraction output for the reason).")


### 11. Final visualization — input vs our predictions vs author reference

One representative acquisition (frame 1): albedo crop (input), our rosette mask and predicted
tracked leaf instances, and the author's provided tracked leaf masks as reference annotation.
**日本語:** 代表的なフレームについて，入力・予測（ロゼット／葉）・著者参照マスクを並べて表示します。

In [ ]:
import glob, os, cv2, numpy as np, matplotlib.pyplot as plt
RAW = sorted(glob.glob("/content/TestData/**/RawData", recursive=True))[0]
OUR = "/content/TestData/OurResults"
sessions = sorted(os.path.basename(d) for d in glob.glob(os.path.join(RAW, "*_NIR")))
s = sessions[0]
rois = np.loadtxt(glob.glob("/content/TestData/**/roi.txt", recursive=True)[0], dtype=np.uint16)
roi = rois[0]
o = np.load(os.path.join(OUR, s, "SNZShadowImAndAlbedo_adaptiveLS.npz"))
alb = o["A"][roi[1]:roi[1]+roi[3], roi[0]:roi[0]+roi[2]]
ours_t = cv2.imread(os.path.join(OUR, "TrackedLeafMasks", "0_%s_mask.png" % s))
prov_t = cv2.imread(sorted(glob.glob(os.path.join(RAW, "TrackedLeafMasks", "*.png")))[0])
rmask = cv2.imread(os.path.join(OUR, "RosetteMasks", "0_%s_mask.png" % s), cv2.IMREAD_GRAYSCALE)
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
axes[0].imshow(alb, cmap="gray"); axes[0].set_title("Input: albedo (our PS solve), %s" % s, fontsize=9)
axes[1].imshow(rmask, cmap="gray"); axes[1].set_title("Our rosette mask (prediction)", fontsize=9)
axes[2].imshow(ours_t); axes[2].set_title("Our tracked leaf instances (prediction)", fontsize=9)
axes[3].imshow(prov_t); axes[3].set_title("Author-provided tracked leaf masks (reference)", fontsize=9)
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()


## Interpretation, differences from the paper, and validation record

- **What was validated:** the author's pinned software (commit `ff1c44b`) executed end-to-end on
  the supplied test acquisitions for Plant 0 in a fresh Colab CPU runtime, with each product
  compared to the author's expected results (adaptive LSV arrays, per-session NPZ arrays,
  rosette masks, leaf instance masks, tracked masks, trait CSVs).
- **What was not verified:** dataset-level accuracies of the paper (SBD/FBD scores, 3D-vs-2D
  error statistics, environment-matrix experiments), training of the networks, and the VIS/VISNIR
  acquisition paths. A single-plant 10-frame run cannot verify population-level claims.
- **Documented adaptations:** CPU TensorFlow 1.15.5 (GPU TF1 needs CUDA 10.0); PyQt4 GUIs replaced
  by their computational cores; POSIX filename fixes; `float64→uint8` cast before PNG write;
  two behavior-preserving vectorizations validated against the author's per-pixel functions.
- **環境・実行の記録:** 実行日時・各ステップの計測時間は各セルの出力に記録されています。無料枠 Colab の
  性能・可用性は変動します。

**Validation record:** filled after the final fresh-runtime run — see `report.md` in the
reproduction repository and the execution outputs preserved in this notebook.
